# T2 AlphaMix+ — O1 재현 실행(시드 5 × 3단계 = 15회) + O5 동적 점검

**사전 등록.** 기준 `notes/prereg_reeval_v3.md` 2·3장, 구현 `notes/alphamix_t2_o1_prereg.md`, 실행기 `replication/alphamix_t2.py` @ `e2cfe8f`(sha256 확인). 결과를 보기 전에 커밋한 노트북이다.

**공식 코드.** `TradeMaster-NTU/PRUDEX-Compass` @ `2b62831` `AlphaMix+/PM`. 파일럿과 같은 호환 shim 4개, 같은 variant.

**순서.** A 데이터 스냅샷(처음 1회 Yahoo 다운로드 → Drive 고정) → B O5 점검(CPU, 1분 내외) → C 학습 15회(1회 약 20분, T4 GPU, 합계 약 5시간) → D 결과 묶음.

**런타임: GPU(T4).** 끊기면 런타임을 다시 연결하고 **모두 실행**하면 완료된 실행은 건너뛴다. 결과는 Drive `data/alphamix_t2/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pathlib
OUT = pathlib.Path('/content/drive/MyDrive/data/alphamix_t2'); OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
COMMIT, SCRIPT_COMMIT = '2b62831', 'e2cfe8f'
SCRIPT_SHA = 'bba444bfd51492658595dfb8b4b1f7e5773847b2230705a690129c97a9389f65'
!test -d /content/PRUDEX || git clone -q https://github.com/TradeMaster-NTU/PRUDEX-Compass.git /content/PRUDEX
!cd /content/PRUDEX && git checkout -q $COMMIT && git log -1 --format='%H %cd'
!pip install -q gym==0.26.2 gtimer exchange_calendars stockstats yfinance python-dateutil cloudpickle
!wget -q -O /content/alphamix_t2.py https://raw.githubusercontent.com/winbean-kwon/ToBeRich/$SCRIPT_COMMIT/replication/alphamix_t2.py
import hashlib; got = hashlib.sha256(open('/content/alphamix_t2.py', 'rb').read()).hexdigest()
assert got == SCRIPT_SHA, ('실행기 해시 불일치', got)
print('실행기 확인', SCRIPT_COMMIT, got[:12])

In [ ]:
# 호환 shim — 파일럿 노트북과 같은 4개(pandas append, yfinance 형식, gym seed, GPU 없을 때 CPU). 하위 프로세스에도 적용되도록 sitecustomize로 넣는다
SHIM = r'''
import pandas as pd
if not hasattr(pd.DataFrame, "append"):
    def _append(self, other, ignore_index=False, **kw):
        return pd.concat([self, other], ignore_index=ignore_index)
    pd.DataFrame.append = _append
try:
    import yfinance as yf
    _dl = yf.download
    def _download(tic, *a, **kw):
        kw.setdefault("auto_adjust", False); kw.setdefault("progress", False)
        df = _dl(tic, *a, **kw)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        return df[["Open", "High", "Low", "Close", "Adj Close", "Volume"]]
    yf.download = _download
except ImportError:
    pass
try:
    import gym
    if not hasattr(gym.Env, "seed"):
        gym.Env.seed = lambda self, seed=None: self._seed(seed) if hasattr(self, "_seed") else [seed]
except ImportError:
    pass
'''
SITE = pathlib.Path('/content/shim'); SITE.mkdir(exist_ok=True); (SITE / 'sitecustomize.py').write_text(SHIM)
import os, subprocess, sys, torch, platform, json
ENVV = dict(os.environ, PYTHONPATH='/content/shim:/content/PRUDEX/AlphaMix+/PM', T2_OUT=str(OUT))
LOGS = OUT / 'logs'; LOGS.mkdir(exist_ok=True)
def sh(args, tag):
    p = subprocess.run([sys.executable, '/content/alphamix_t2.py'] + args, capture_output=True, text=True, env=ENVV,
                       cwd='/content/PRUDEX/AlphaMix+/PM')
    (LOGS / f'{tag}.log').write_text(p.stdout + '\n--- stderr ---\n' + p.stderr)
    if p.returncode != 0:
        print(p.stdout[-2000:]); print('stderr 끝부분:', p.stderr[-3000:]); raise RuntimeError(f'{tag} 실패 — 로그 {LOGS}/{tag}.log')
    return p
env_info = dict(python=platform.python_version(), torch=torch.__version__, cuda=torch.cuda.is_available(),
                gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
(OUT / 'env_info.json').write_text(json.dumps(env_info)); print(env_info)
assert env_info['cuda'], 'GPU 런타임이 아닙니다 — 런타임 유형을 T4 GPU로 바꾸세요'

## A. 데이터 스냅샷 — 처음 1회만 Yahoo에서 받아 고정(이후 해시 확인 후 그 파일만 사용)

In [ ]:
%%time
if (OUT / 'snapshot' / 'splits.json').exists():
    print('스냅샷 있음 — 건너뜀')
else:
    sh(['snap'], 'A_snap')
print((OUT / 'snapshot' / 'meta.json').read_text()[:600])
print(json.dumps({k: {s: v[s] for s in ('train', 'valid', 'test')} for k, v in json.loads((OUT / 'snapshot' / 'splits.json').read_text()).items()}, indent=1))

## B. O5 동적 점검(C1·C2·C3·S1) — CPU, 학습 없음

In [ ]:
if not (OUT / 'o5' / 'o5_result.json').exists():
    sh(['o5'], 'B_o5')
r = json.loads((OUT / 'o5' / 'o5_result.json').read_text())
for k, v in r.items():
    print(k, {a: b for a, b in v.items() if a.startswith(('C1', 'C2', 'S1_measured_over'))} if k != 'C3' else v)

## C. O1 학습 15회 — 단계 S2021 → S2020 → S2019, 시드 순(완료된 실행은 건너뜀)

In [ ]:
import time
STAGES = ['S2021', 'S2020', 'S2019']; SEEDS = [12345, 23451, 34512, 45123, 51234]
for st in STAGES:
    for s in SEEDS:
        d = OUT / 'o1' / f'{st}_s{s}'
        if (d / 'meta.json').exists():
            print(st, s, '완료됨'); continue
        t0 = time.time(); sh(['train', st, str(s)], f'C_{st}_s{s}')
        tr = __import__('pandas').read_csv(d / 'result.csv')['test_tr'].iloc[-1]
        print(f'{st} {s}: {(time.time() - t0) / 60:.1f}분, 마지막 행 test_tr {tr:.4f}', time.strftime('%H:%M'), flush=True)
print('15회 모두 완료' if all((OUT / 'o1' / f'{st}_s{s}' / 'meta.json').exists() for st in STAGES for s in SEEDS) else '미완료 있음')

## D. 결과 묶음 — 끝나면 Claude에게 "끝났어"라고 알려 주세요(Drive `data/alphamix_t2/t2_o1_bundle.zip`)

In [ ]:
import shutil
z = shutil.make_archive('/content/t2_o1_bundle', 'zip', OUT, '.')
shutil.copy(z, OUT / 't2_o1_bundle.zip'); print(OUT / 't2_o1_bundle.zip', round(os.path.getsize(z) / 1e6, 1), 'MB')